In [1]:
!nvidia-smi

Sat Mar  7 11:32:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from google.colab import drive
drive.mount('/content/drive')

!cp /content/gdrive/MyDrive/path/to/data.zip /content

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cp: cannot stat '/content/gdrive/MyDrive/path/to/resnet_croped_train_dataset.zip': No such file or directory


In [6]:
!unzip -q /content/data.zip -d /content/custom_data

In [ ]:
!wget -O /content/train_val_split.py https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/utils/train_val_split.py

# TO DO: Improve robustness of train_val_split.py script so it can handle nested data folders, etc
!python train_val_split.py --datapath="/content/custom_data" --train_pct=0.9


--2026-03-02 11:59:49--  https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/utils/train_val_split.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3203 (3.1K) [text/plain]
Saving to: ‘/content/train_val_split.py’

/content/train_val_ 100%[===================>]   3.13K  --.-KB/s    in 0s      

2026-03-02 11:59:49 (38.4 MB/s) - ‘/content/train_val_split.py’ saved [3203/3203]

Created folder at /content/data/train/images.
Created folder at /content/data/train/labels.
Created folder at /content/data/validation/images.
Created folder at /content/data/validation/labels.
Number of image files: 993
Number of annotation files: 993
Images moving to train: 893
Images moving to validation: 100


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.3 MB/s eta 0:00:00


In [ ]:
# Python function to automatically create data.yaml config file
# 1. Reads "classes.txt" file to get list of class names
# 2. Creates data dictionary with correct paths to folders, number of classes, and names of classes
# 3. Writes data in YAML format to data.yaml

import yaml
import os

def create_data_yaml(path_to_classes_txt, path_to_data_yaml):

    # Read class.txt to get class names
    if not os.path.exists(path_to_classes_txt):
        print(f'classes.txt file not found! Please create a classes.txt labelmap and move it to {path_to_classes_txt}')
        return
    with open(path_to_classes_txt, 'r') as f:
        classes = []
        for line in f.readlines():
            if len(line.strip()) == 0: continue
            classes.append(line.strip())
    number_of_classes = len(classes)

    # Create data dictionary
    data = {
        'path': '/content/data',
        'train': 'train/images',
        'val': 'validation/images',
        'nc': number_of_classes,
        'names': classes
    }

    # Write data to YAML file
    with open(path_to_data_yaml, 'w') as f:
        yaml.dump(data, f, sort_keys=False)
    print(f'Created config file at {path_to_data_yaml}')

    return

# Define path to classes.txt and run function
path_to_classes_txt = '/content/custom_data/classes.txt'
path_to_data_yaml = '/content/data.yaml'

create_data_yaml(path_to_classes_txt, path_to_data_yaml)

print('\nFile contents:\n')
!cat /content/data.yaml


Created config file at /content/data.yaml

File contents:

path: /content/data
train: train/images
val: validation/images
nc: 2
names:
- Buffalo
- Cow


In [ ]:
!yolo detect train data=/content/data.yaml model=yolov26.pt epochs=150 imgsz=720

In [ ]:
!zip -r predict.zip /content/runs/detect/predict

  adding: content/runs/detect/train4/ (stored 0%)
  adding: content/runs/detect/train4/BoxP_curve.png (deflated 17%)
  adding: content/runs/detect/train4/val_batch1_pred.jpg (deflated 12%)
  adding: content/runs/detect/train4/val_batch0_pred.jpg (deflated 10%)
  adding: content/runs/detect/train4/results.csv (deflated 63%)
  adding: content/runs/detect/train4/confusion_matrix.png (deflated 35%)
  adding: content/runs/detect/train4/BoxPR_curve.png (deflated 20%)
  adding: content/runs/detect/train4/train_batch2.jpg (deflated 5%)
  adding: content/runs/detect/train4/train_batch0.jpg (deflated 5%)
  adding: content/runs/detect/train4/val_batch0_labels.jpg (deflated 10%)
  adding: content/runs/detect/train4/val_batch1_labels.jpg (deflated 13%)
  adding: content/runs/detect/train4/results.png (deflated 8%)
  adding: content/runs/detect/train4/BoxR_curve.png (deflated 14%)
  adding: content/runs/detect/train4/labels.jpg (deflated 24%)
  adding: content/runs/detect/train4/val_batch2_pred.jpg 

In [ ]:
from google.colab import files
files.download("predict.zip")


FileNotFoundError: Cannot find file: predict.zip

In [ ]:
from ultralytics import YOLO

# Load your best model
model = YOLO("best.pt")

# Run inference on an image
results = model.predict(source="HomeFires.jpg", show=True, save=True, conf=0.25)


FileNotFoundError: [Errno 2] No such file or directory: 'best.pt'

In [ ]:
from ultralytics import YOLO

model = YOLO("best.pt")

# Run inference on a video
results = model.predict(source="test_video.mp4", show=True, save=True, conf=0.25)


In [ ]:
from ultralytics import YOLO

# Load your trained model
model = YOLO("best.pt")

# Run inference on all images & videos in the folder
results = model.predict(
    source="/content/fire_test_folder",  # folder with .jpg, .png, .mp4 etc.
    show=True,     # show in interactive window (works locally, in Colab it won't popup)
    save=True,     # saves output with detections
    conf=0.4       # confidence threshold
)